In [24]:
from pathlib import Path
import pandas as pd
import numpy as np


DATA_DIR = Path(
    r"C:\Python\CSUREMM\data_primitive"
)


gt_diff = pd.read_csv(
    DATA_DIR / "gt_diff_panel.csv",
    parse_dates=["Time"]
).set_index("Time")


gt_logdiff = pd.read_csv(
    DATA_DIR / "gt_log_diff_panel.csv",
    parse_dates=["Time"]
).set_index("Time")


stationarity = pd.read_csv(
    DATA_DIR / "stationarity_summary.csv"
)


sp500 = pd.read_csv(
    DATA_DIR / "sp500_daily.csv",
    parse_dates=["date"]
).set_index("date")

In [25]:
def gt_eda_summary(panel):

    rows = []

    for col in panel.columns:

        x = panel[col]

        rows.append({

            "term": col,

            "observations":
                x.notna().sum(),

            "missing":
                x.isna().sum(),

            "mean":
                x.mean(),

            "std":
                x.std(),

            "min":
                x.min(),

            "max":
                x.max(),

            "zero_share":
                (x == 0).mean(),

            "unique_values":
                x.nunique(),

            "abs_change_mean":
                x.abs().mean(),

            "abs_change_std":
                x.abs().std()

        })


    return (
        pd.DataFrame(rows)
        .sort_values(
            "std",
            ascending=False
        )
    )


eda = gt_eda_summary(
    gt_logdiff
)


eda.to_csv(
    DATA_DIR / "eda_summary.csv",
    index=False
)


print(
    eda.head(20)
)

             term  observations  missing          mean       std       min  \
131      squander          1611        0  3.307927e-18  3.096515 -6.447306   
22       bequeath          1611        0 -2.713446e-03  2.981310 -6.366470   
24      betrothal          1611        0  0.000000e+00  2.429366 -6.366470   
2       affluence          1611        0  2.348328e-03  2.422803 -5.755742   
96       nobleman          1611        0  2.637816e-04  2.275759 -5.247024   
92    meritorious          1611        0  2.714677e-03  2.220432 -5.646624   
143  unprofitable          1611        0  1.102642e-18  2.099245 -6.413459   
4          afloat          1611        0  3.021581e-04  1.949352 -5.371185   
88      liquidate          1611        0  2.383895e-03  1.778331 -5.469548   
124      richness          1611        0  4.012135e-04  1.692590 -4.892019   
54      destitute          1611        0 -5.898476e-05  1.567058 -4.958944   
8    aristocratic          1611        0  2.171385e-03  1.543731

In [26]:
corr = (
    gt_logdiff
    .corr()
)


corr.to_csv(
    DATA_DIR / "corr_matrix.csv"
)


print(
    corr.iloc[:10,:10]
)

                accrue  advantage  affluence  affluent    afloat  allowance  \
accrue        1.000000   0.690533   0.270753  0.336467  0.171432   0.550716   
advantage     0.690533   1.000000   0.353016  0.502309  0.239729   0.755997   
affluence     0.270753   0.353016   1.000000  0.271722  0.159937   0.333974   
affluent      0.336467   0.502309   0.271722  1.000000  0.116125   0.456373   
afloat        0.171432   0.239729   0.159937  0.116125  1.000000   0.208211   
allowance     0.550716   0.755997   0.333974  0.456373  0.208211   1.000000   
aristocracy   0.401703   0.463856   0.235625  0.296572  0.203201   0.403888   
aristocrat    0.192176   0.288323   0.123998  0.181889  0.127261   0.260926   
aristocratic  0.188793   0.254977   0.148025  0.175292  0.055280   0.194755   
associate     0.698664   0.918880   0.356604  0.497372  0.234860   0.773968   

              aristocracy  aristocrat  aristocratic  associate  
accrue           0.401703    0.192176      0.188793   0.698664  


In [27]:
def high_corr_pairs(
    corr,
    threshold=0.8
):

    pairs=[]

    cols = corr.columns


    for i in range(len(cols)):

        for j in range(i+1,len(cols)):

            value = corr.iloc[i,j]

            if abs(value) >= threshold:

                pairs.append({

                    "term1":cols[i],

                    "term2":cols[j],

                    "correlation":value

                })


    return (
        pd.DataFrame(pairs)
        .sort_values(
            "correlation",
            key=lambda x:x.abs(),
            ascending=False
        )
    )


corr_pairs = high_corr_pairs(
    corr,
    0.8
)


corr_pairs.to_csv(
    DATA_DIR / "corr_high_pairs.csv",
    index=False
)


print(
    corr_pairs.head(20)
)

            term1         term2  correlation
208    contribute    successful     0.942629
155     community   cooperative     0.935522
34      associate     community     0.932570
6       advantage     community     0.932538
182  compensation   partnership     0.932338
163     community   partnership     0.930445
251       default   partnership     0.930026
46      associate   partnership     0.928346
229   cooperative   partnership     0.925440
35      associate  compensation     0.923936
174  compensation   cooperative     0.922704
152     community  compensation     0.922377
30      associate   beneficiary     0.921875
178  compensation        equity     0.921644
76    beneficiary     community     0.918984
154     community  contribution     0.918883
0       advantage     associate     0.918880
213  contribution       expense     0.917737
38      associate   cooperative     0.914800
97        benefit    contribute     0.914756


Question: Are there latent dimensions in search behavior that correspond to Fear and Greed?

** Section 1: PCA **

In [28]:
# Standardize the data for factor analysis

from sklearn.preprocessing import StandardScaler

X = gt_logdiff.copy()

# remove columns with too many missing values
valid_cols = (
    X.isna()
    .mean()
    .loc[lambda x: x < 0.10]
    .index
)

X = X[valid_cols]

# simple imputation
X = X.fillna(0)

scaler = StandardScaler()

X_std = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

In [29]:
from sklearn.decomposition import PCA

pca = PCA()

pca.fit(X_std)

explained = pca.explained_variance_ratio_

explained[:10]

array([0.34106179, 0.05111192, 0.03356053, 0.01370481, 0.0110737 ,
       0.01051435, 0.00957161, 0.00925633, 0.00909347, 0.00891228])

In [30]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=X_std.columns,
    columns=[
        f"PC{i+1}"
        for i in range(len(X_std.columns))
    ]
)

loadings.to_csv(
    DATA_DIR / "pca_loadings.csv"
)

for pc in ["PC1","PC2","PC3"]:

    print("\n", pc)

    print(
        loadings[pc]
        .sort_values()
        .tail(15)
    )

    print(
        loadings[pc]
        .sort_values()
        .head(15)
    )


 PC1
success         0.128553
successful      0.130389
capitalize      0.130673
contribution    0.130707
default         0.130798
expense         0.130927
cooperative     0.131076
profit          0.131167
compensation    0.131219
associate       0.131255
advantage       0.131442
benefit         0.132445
community       0.132707
partnership     0.133545
contribute      0.134202
Name: PC1, dtype: float64
treasure     -0.097488
thrift       -0.094377
redemption   -0.094252
buy          -0.087463
bum          -0.085456
worth        -0.079927
race         -0.074545
patron       -0.071712
hustler      -0.066654
gift         -0.066193
cheap        -0.063455
luxury       -0.062220
hole         -0.056780
ghetto       -0.055199
vagabond     -0.054000
Name: PC1, dtype: float64

 PC2
donate         0.151697
donation       0.151865
lay            0.152214
bum            0.152732
fine           0.160061
broke          0.162830
expensive      0.177394
cheap          0.180518
gold           0.180689


** Section 2 ** Factor Analysis

In [31]:
from sklearn.decomposition import FactorAnalysis

fa = FactorAnalysis(
    n_components=3,
    random_state=42,
    rotation="varimax" #added this to make the results more interpretable
)

scores = fa.fit_transform(X_std)

loadings = pd.DataFrame(
    fa.components_.T,
    index=X_std.columns,
    columns=[
        "Factor1",
        "Factor2",
        "Factor3"
    ]
)

print(loadings)

# loadings.to_csv("factor_loadings.csv")

            Factor1   Factor2   Factor3
accrue     0.706732 -0.009530  0.095742
advantage  0.947690 -0.006075  0.111168
affluence  0.389209 -0.116021 -0.103056
affluent   0.551309 -0.048596 -0.163128
afloat     0.266691 -0.014223 -0.061298
...             ...       ...       ...
vagrant    0.255317 -0.005696 -0.074490
valuable   0.344816  0.037662 -0.470458
warfare   -0.230899  0.124331 -0.297063
waste      0.569468  0.322655  0.362945
worth     -0.548613  0.304064 -0.299264

[150 rows x 3 columns]


In [32]:
# 1. Calculate the sum of squares of loadings for each factor
ss_loadings = (loadings ** 2).sum()

# 2. Proportion of variance explained by each factor
variance_explained = ss_loadings / len(X_std.columns)

# 3. Print the explained variance ratio and cumulative variance
print("Variance Explained by Each Factor:")
print(variance_explained)
print("\nCumulative Variance Explained:")
print(variance_explained.cumsum())

Variance Explained by Each Factor:
Factor1    0.336311
Factor2    0.043042
Factor3    0.035577
dtype: float64

Cumulative Variance Explained:
Factor1    0.336311
Factor2    0.379353
Factor3    0.414930
dtype: float64


In [33]:
# Create factor scores DataFrame

factor_scores = pd.DataFrame(
    scores,
    index=X_std.index,
    columns=[
        "Factor1",
        "Factor2",
        "Factor3"
    ]
)

factor_scores

,Factor1,Factor2,Factor3
Time,,,
2022-01-02,0.439294,0.697160,-1.143778
2022-01-03,0.085540,-0.811563,1.108541
2022-01-04,1.146483,2.162181,-2.225281
2022-01-05,-0.205586,-0.754178,0.326958
2022-01-06,-1.158643,-4.425895,2.413040
...,...,...,...
2026-05-27,0.197479,0.571040,0.476888
2026-05-28,0.201478,0.674566,-0.648154
2026-05-29,-0.356899,-0.308402,0.190454


The Google Trends universe contains one dominant common
attention factor explaining approximately 34% of total variation.

Additional factors exist but are much weaker and appear
to reflect heterogeneous themes such as consumer spending,
social cooperation, and economic distress.

A clean Fear-Greed decomposition has not yet emerged.

In [34]:
# Step 1: Build a daily Factor 1 baseline index from factor_scores
# Assumes factor_scores.index is a DatetimeIndex (inherited from gt_logdiff's index)

import pandas as pd

# Ensure the index is datetime (in case it loaded as strings/objects)
factor_scores.index = pd.to_datetime(factor_scores.index)

# Extract Factor 1 as the baseline index
factor1_index = factor_scores["Factor1"].copy()
factor1_index.name = "Factor1_Index"

# Sort chronologically just in case
factor1_index = factor1_index.sort_index()

# Optional: z-score normalize so it's comparable in scale to VIX/EPU/F&G later
factor1_index_z = (
    (factor1_index - factor1_index.mean()) / factor1_index.std()
)
factor1_index_z.name = "Factor1_Index_Z"

# Optional: smooth with a rolling average (e.g. 5-day) to reduce day-to-day noise
factor1_index_smoothed = factor1_index_z.rolling(window=5, min_periods=1).mean()
factor1_index_smoothed.name = "Factor1_Index_Smoothed"

# Combine into one DataFrame for easy plotting later
baseline_df = pd.DataFrame({
    "Factor1_raw": factor1_index,
    "Factor1_z": factor1_index_z,
    "Factor1_smoothed": factor1_index_smoothed
})

baseline_df.head()

,Factor1_raw,Factor1_z,Factor1_smoothed
Time,,,
2022-01-02,0.439294,0.439963,0.439963
2022-01-03,0.085540,0.085671,0.262817
2022-01-04,1.146483,1.148231,0.557955
2022-01-05,-0.205586,-0.205900,0.366992
2022-01-06,-1.158643,-1.160410,0.061511


In [35]:
# Step 2: use all three factors and use variance weighting

import pandas as pd

# Ensure the index is datetime (in case it loaded as strings/objects)
factor_scores.index = pd.to_datetime(factor_scores.index)

# Sort chronologically just in case
factor_scores = factor_scores.sort_index()

baseline_df = pd.DataFrame(index=factor_scores.index)

for factor in ["Factor1", "Factor2", "Factor3"]:

    raw = factor_scores[factor].copy()
    raw.name = f"{factor}_raw"

    # z-score normalize (comparable scale across factors and later vs. VIX/EPU/F&G)
    z = (raw - raw.mean()) / raw.std()
    z.name = f"{factor}_z"

    # Factor scores are daily differences (derived from gt_logdiff), not levels.
    # Cumulatively sum so each factor becomes a running INDEX/LEVEL series,
    # comparable to level-based series like VIX/EPU/Fear&Greed.
    cumulative = z.cumsum()
    cumulative.name = f"{factor}_cumulative"

    # Light rolling smoothing on the cumulative index, mainly for visual
    # de-jaggedness (not noise removal, since cumsum already removes most of that)
    smoothed = cumulative.rolling(window=10, min_periods=1, center=True).mean()
    smoothed.name = f"{factor}_smoothed"

    baseline_df[raw.name] = raw
    baseline_df[z.name] = z
    baseline_df[cumulative.name] = cumulative
    baseline_df[smoothed.name] = smoothed

baseline_df.head()

,Factor1_raw,Factor1_z,Factor1_cumulative,Factor1_smoothed,Factor2_raw,Factor2_z,Factor2_cumulative,Factor2_smoothed,Factor3_raw,Factor3_z,Factor3_cumulative,Factor3_smoothed
Time,,,,,,,,,,,,
2022-01-02,0.439294,0.439963,0.439963,0.882997,0.697160,0.721522,0.721522,0.163902,-1.143778,-1.185911,-1.185911,-1.014776
2022-01-03,0.085540,0.085671,0.525634,0.869932,-0.811563,-0.839923,-0.118401,0.010635,1.108541,1.149376,-0.036535,-1.095183
2022-01-04,1.146483,1.148231,1.673866,0.647804,2.162181,2.237739,2.119338,0.113626,-2.225281,-2.307254,-2.343789,-1.280751
2022-01-05,-0.205586,-0.205900,1.467966,0.445648,-0.754178,-0.780533,1.338805,0.056966,0.326958,0.339002,-2.004787,-1.580862
2022-01-06,-1.158643,-1.160410,0.307556,0.447027,-4.425895,-4.580558,-3.241753,-0.149288,2.413040,2.501930,0.497143,-1.431709


In [36]:
weights = pd.Series({"Factor1": 0.336311, "Factor2": 0.043042, "Factor3": 0.035577})
weights = weights / weights.sum()  # normalize to sum to 1

baseline_df["Composite_Index"] = sum(
    weights[f] * baseline_df[f"{f}_cumulative"] for f in ["Factor1", "Factor2", "Factor3"]
)

In [37]:
baseline_df.to_csv(DATA_DIR / "baseline_index.csv", index=True)